In [17]:
import sys
sys.path.append('../')

import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
from tqdm import tqdm
from matplotlib.colors import LogNorm
import pytz, cmath, itertools
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import scipy.sparse as ssp
from sympy import symbols
import utils_2Q_gate_zp as ut
import pandas as pd
import scipy as sp
from multiprocessing import Pool
import qutip as qt
import multiprocessing as mp
from multiprocessing import Pool
import scqubits as scq
from sympy import symbols
import scipy.sparse as ssp
from datetime import datetime

In [22]:
Ec0=1.0
truc1=10
truc_tot=50
charge_pick=False
n_cut=20
phi_cut=30

zp = scq.Circuit(ut.zp_yml, from_file=False)
zp.Ec0 = Ec0
zp.configure(transformation_matrix=np.linalg.inv(ut.transform_2zeropi))

##############################################################################################
### Construct subsystem, calculate eigenvalues
system_hierarchy = [[1,2],  [5,6]]
subsystem_trunc_dims = [10, 10]
zp.configure(system_hierarchy=system_hierarchy,
            subsystem_trunc_dims=subsystem_trunc_dims)
zp.Φ1 = 0.001
zp.Φ2 = 0.001
zp.cutoff_ext_1, zp.cutoff_ext_5 = phi_cut, phi_cut
zp.cutoff_n_2, zp.cutoff_n_6 = n_cut, n_cut

### the two-line code below takes time when truc1 is large
eval0, _ = zp.subsystems[0].eigensys(evals_count=truc1)
eval1, _ = zp.subsystems[1].eigensys(evals_count=truc1)

sorted_idx0 = np.argsort(eval0)
eval0 = eval0[sorted_idx0]
eval0 = eval0 - eval0[0]
sorted_idx1 = np.argsort(eval1)
eval1 = eval1[sorted_idx1]
eval1 = eval1 - eval1[0]

In [23]:
print('eval0:', eval0)
print('eval1:', eval1)

eval0: [0.         2.47840671 2.7750845  2.77568787 3.61880049 4.84356392
 5.42174571 5.42471491 6.05145361 7.06230103]
eval1: [0.         2.34149707 2.91159299 2.91328107 3.59467162 4.56551433
 5.37440738 5.38091609 5.8791023  6.63683681]


In [ ]:
print('eval0:', eval0)
print('eval1:', eval1)

eval0: [0.         2.47838081 2.77501524 2.77553849 3.61855135 4.84351416
 5.4222014  5.4240631  6.05125379 7.06222765]
eval1: [0.         2.34147571 2.91204612 2.91266483 3.59447119 4.56547274
 5.37515925 5.38002149 5.87894762 6.63677356]


In [20]:
zp.sym_external_fluxes()

{Φ1: (Branch(JJ, 3, 4, id_str: 1),
  [Branch(JJ, 1, 2, id_str: 0),
   Branch(L, 1, 4, id_str: 3),
   Branch(JJ, 3, 4, id_str: 1),
   Branch(L, 2, 3, id_str: 2)]),
 Φ2: (Branch(JJ, 7, 8, id_str: 7),
  [Branch(JJ, 5, 6, id_str: 6),
   Branch(L, 5, 8, id_str: 9),
   Branch(JJ, 7, 8, id_str: 7),
   Branch(L, 6, 7, id_str: 8)])}

In [ ]:
folder = f'../../data/3ncut_two_zeropi/truc1=500/'
n_theta0 = np.load(folder+'n_theta0.npy')
n_theta1 = np.load(folder+'n_theta1.npy')
eval0 = pd.read_csv(folder+ 'eval0.txt').to_numpy().flatten()
eval1 = pd.read_csv(folder+ 'eval1.txt').to_numpy().flatten()

t1_other = 2 # μs
gamma_decay_logi =  1 / 1600e3
gamma_dephase_logi = 1 / 100e3
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

In [7]:
truc1, truc_tot, charge_pick = 300, 1000, True
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten()
hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
print('len(hspace_0)=', len(hspace_0))
print('len(hspace_1)=', len(hspace_1))
print('hspace_0=', hspace_0)
print('hspace_1=', hspace_1)

len(hspace_0)= 154
len(hspace_1)= 155
hspace_0= [  0   1   2   4   5   8   9  12  13  15  18  20  22  24  25  26  28  30
  33  34  35  37  39  41  44  45  46  50  51  54  56  58  59  60  64  65
  66  69  70  74  75  77  79  81  84  85  86  89  90  92  93  98  99 100
 101 102 105 106 109 110 111 115 116 121 123 124 127 128 129 131 132 135
 136 138 140 141 144 145 147 150 151 154 157 160 161 162 165 167 169 170
 173 174 175 177 179 180 183 185 188 189 194 195 198 199 200 201 204 205
 207 208 210 213 214 216 218 222 223 224 225 228 229 234 235 238 241 244
 247 248 249 250 251 252 254 255 260 261 262 263 264 265 269 270 271 275
 277 281 282 286 288 289 293 294 295 299]
hspace_1= [  0   1   2   4   5   8   9  12  13  16  18  20  21  24  25  26  28  30
  33  34  35  36  39  42  44  45  46  52  53  54  55  57  59  60  64  65
  66  68  70  73  76  77  78  81  82  83  86  89  90  92  93  97  98 100
 102 103 105 106 109 110 112 114 116 121 123 124 125 128 129 131 132 135
 136 138 140 141 144 145

### $T_1$

$\gamma_{ll'} = \Gamma | \bra{l} n_\theta \ket{l'} |^2 = 1/ T_1  $ ( $T_1$ is intrawell decay time )

$\gamma_{07} = \Gamma_{07} | \bra{0} n_\theta \ket{7} |^2 = 1 / (2\times 10^{-6})   $

In [13]:
drive_qubit_0 = False
hspace = hspace_0 if drive_qubit_0 else hspace_1
n_theta = n_theta0 if drive_qubit_0 else n_theta1

idx_2 = hspace.tolist().index(2)
gamma_decay_0 = []
gamma_decay_2 = []
Gamma_0 = gamma_decay_other / (n_theta0[0,8]**2)
Gamma_2 = gamma_decay_other / (n_theta0[2,8]**2)
for i in hspace:
    gamma_decay_0.append(Gamma_0* n_theta[0, i]**2)
    gamma_decay_2.append(Gamma_2* n_theta[0, i]**2)

gamma_decay_0[idx_2] = gamma_decay_logi
gamma_decay_2[idx_2] = gamma_decay_logi
print('np.imag(gamma_decay_0)=0 -->', np.all(np.imag(gamma_decay_0)==0))
print('np.imag(gamma_decay_2)=0 -->', np.all(np.imag(gamma_decay_2)==0))

if np.all(np.imag(gamma_decay_0)==0) and np.all(np.imag(gamma_decay_2)==0):
    gamma_decay_0 = np.real(gamma_decay_0)
    gamma_decay_2 = np.real(gamma_decay_2)
print('t1_other = %d μs'%t1_other)
print('gamma_decay_0=', gamma_decay_0.tolist())
print('gamma_decay_2=', gamma_decay_2.tolist())


np.imag(gamma_decay_0)=0 --> True
np.imag(gamma_decay_2)=0 --> True
t1_other = 50 μs
gamma_decay_0= [0.0, 0.017984010291838794, 6.25e-07, 0.0, 6.077609517275637e-07, 2.2242440785374412e-05, 0.0, 0.0, 0.0, 1.1025577389795072e-07, 3.3609931171352424e-07, 0.0, 2.8832207738871286e-08, 0.0, 0.0, 2.7724141593293816e-05, 0.0, 0.0, 6.341306543441377e-06, 4.6855942753518475e-06, 0.0, 0.0, 8.945921308927219e-11, 3.9172357746172246e-09, 0.0, 9.329180318109473e-07, 0.0, 5.014318666980138e-07, 0.0, 0.0, 2.200883889804547e-07, 0.0, 1.300165584609364e-08, 0.0, 0.0, 2.6201633663609747e-08, 0.0, 9.31457712802099e-08, 0.0, 6.465564748231819e-09, 0.0, 5.19227470084653e-08, 0.0, 1.2001479765297154e-07, 0.0, 1.2345957319162472e-07, 0.0, 6.856845029528533e-10, 0.0, 0.0, 1.5266139901470675e-09, 0.0, 2.1697443361685333e-08, 0.0, 2.4258068721294135e-08, 0.0, 4.907575121330216e-09, 0.0, 3.7147116167393674e-08, 0.0, 2.3808438498569898e-09, 0.0, 6.045733784804644e-10, 0.0, 2.546095821401587e-09, 0.0, 2.5039817580

In [14]:
n_theta1_decay_0 = gamma_decay_0
n_theta1_decay_2 = gamma_decay_2

In [16]:
print('n_theta1_decay_0=', n_theta1_decay_0.tolist())
print('n_theta1_decay_2=', n_theta1_decay_2.tolist())
len(n_theta1_decay_0)

n_theta1_decay_0= [0.0, 0.017984010291838794, 6.25e-07, 0.0, 6.077609517275637e-07, 2.2242440785374412e-05, 0.0, 0.0, 0.0, 1.1025577389795072e-07, 3.3609931171352424e-07, 0.0, 2.8832207738871286e-08, 0.0, 0.0, 2.7724141593293816e-05, 0.0, 0.0, 6.341306543441377e-06, 4.6855942753518475e-06, 0.0, 0.0, 8.945921308927219e-11, 3.9172357746172246e-09, 0.0, 9.329180318109473e-07, 0.0, 5.014318666980138e-07, 0.0, 0.0, 2.200883889804547e-07, 0.0, 1.300165584609364e-08, 0.0, 0.0, 2.6201633663609747e-08, 0.0, 9.31457712802099e-08, 0.0, 6.465564748231819e-09, 0.0, 5.19227470084653e-08, 0.0, 1.2001479765297154e-07, 0.0, 1.2345957319162472e-07, 0.0, 6.856845029528533e-10, 0.0, 0.0, 1.5266139901470675e-09, 0.0, 2.1697443361685333e-08, 0.0, 2.4258068721294135e-08, 0.0, 4.907575121330216e-09, 0.0, 3.7147116167393674e-08, 0.0, 2.3808438498569898e-09, 0.0, 6.045733784804644e-10, 0.0, 2.546095821401587e-09, 0.0, 2.5039817580865508e-09, 0.0, 0.0, 3.946207013636768e-10, 0.0, 3.1926351323941588e-09, 0.0, 7.5

155

In [12]:
print('n_theta0_decay_0=', n_theta0_decay_0.tolist())
print('n_theta0_decay_2=', n_theta0_decay_2.tolist())
len(n_theta0_decay_0)

n_theta0_decay_0= [0.0, 0.019100215738036556, 6.25e-07, 0.0, 2.029578732273843e-07, 1.9999999999999998e-05, 0.0, 0.0, 0.0, 7.929121148094782e-08, 5.3458001764679956e-08, 3.540304633822212e-08, 0.0, 0.0, 0.0, 2.6053512947003472e-05, 0.0, 0.0, 3.94903784248221e-06, 0.0, 6.474965293793219e-06, 0.0, 6.932203256545913e-08, 3.1744281414173326e-08, 0.0, 1.1149231190609815e-06, 0.0, 5.063378104782247e-07, 0.0, 0.0, 1.416721926344015e-07, 0.0, 4.942338954072857e-08, 0.0, 0.0, 1.0941166568821438e-09, 0.0, 7.923538749558526e-08, 0.0, 1.0260394698418787e-09, 0.0, 4.7253689422128595e-08, 0.0, 1.2047625151674405e-07, 0.0, 1.227029825199256e-07, 0.0, 1.3211826531980858e-09, 0.0, 0.0, 4.744317458955279e-10, 0.0, 1.109720939581947e-08, 0.0, 0.0, 4.1871249626358426e-08, 9.060161988126e-09, 0.0, 3.2777796166607045e-08, 0.0, 6.472634487481029e-09, 0.0, 3.2738578765045836e-10, 0.0, 7.601319548826239e-10, 0.0, 4.63411393931922e-09, 0.0, 0.0, 7.967445165816826e-10, 0.0, 8.816338493243842e-10, 0.0, 1.00519650

154

### Noisy fidelity

In [ ]:
truc1, truc_tot, charge_pick = 300, 300, True
truc_tot_2 = 14
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}_eket/'
hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten()
hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta0_dress.txt').to_numpy()
n_theta1_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta1_dress.txt').to_numpy()

cz300_se_3ncut= pd.read_csv('data/data_cz_3ncut_truc1=300_select.txt')
params = cz300_se_3ncut[['tg', 'drive_amp', 'detune']].to_numpy()[:1]

In [22]:
cz300_se_3ncut= pd.read_csv('data/data_cz_3ncut_truc1=300_select.txt')
params = cz300_se_3ncut[['tg', 'drive_amp', 'detune']].to_numpy()
params[[0, -1],:]

array([[2.00471530e+01, 1.94350000e-02, 7.27300000e-03],
       [2.00001076e+02, 6.93600000e-03, 1.20520000e-02]])

In [3]:
eket_tot = pd.read_csv(folder+ 'eket_tot.txt').map(complex).to_numpy()
eket_tot = eket_tot[:truc_tot_2]

In [4]:
dim_0 = len(hspace_0)
dim_1 = len(hspace_1)
eval_tot = eval_tot[:truc_tot_2]
hspace_full = hspace_full[:truc_tot_2]
hspace_dress = np.arange(truc_tot_2)
n_theta0_dress = qt.Qobj(n_theta0_dress[np.ix_(hspace_dress, hspace_dress)])
n_theta1_dress = qt.Qobj(n_theta1_dress[np.ix_(hspace_dress, hspace_dress)])

In [5]:
gamma_decay_logi =  0 / 1600e3
gamma_dephase_logi = 0
gamma_decay_other = 0 / 2e3
gamma_dephase_other = 0 / 400
jump_t1   = []
jump_tphi = []

if charge_pick:
    gamma_decay   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (dim_1-3)
    gamma_dephase = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (dim_1-3)
    qubit_a = True
    args = [dim_0, dim_1, gamma_decay, gamma_dephase, eket_tot, qubit_a]
    jump_op_a = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *args) for state in range(1,dim_0))
    qubit_a = False
    args = [dim_0, dim_1, gamma_decay, gamma_dephase, eket_tot, qubit_a]
    jump_op_b = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *args) for state in range(1,dim_1))
    jump_t1_list = np.array(jump_op_a)[:,0].tolist() + np.array(jump_op_b)[:,0].tolist()
    jump_tphi_list = np.array(jump_op_a)[:,1].tolist() + np.array(jump_op_b)[:,1].tolist()
    jump_t1_list = [qt.Qobj(matrix) for matrix in jump_t1_list]
    jump_tphi_list = [qt.Qobj(matrix) for matrix in jump_tphi_list]
else:
    gamma_decay   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (truc1-3)
    gamma_dephase = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (truc1-3)
    args = [truc1, gamma_decay, gamma_dephase, eket_tot]
    jump_op = Parallel(n_jobs=100)(delayed(ut.get_jump_op)(state, *args) for state in range(1,truc1))
    jump_t1 = np.array(jump_op)[:,:2]
    jump_tphi = np.array(jump_op)[:,2:]
    jump_t1_list = [qt.Qobj(matrix) for row in jump_t1 for matrix in row]
    jump_tphi_list = [qt.Qobj(matrix) for row in jump_tphi for matrix in row]

In [17]:
logi_state = ['0-0', '0-2', '2-0', '2-2']
W_20_50 = np.abs(eval_tot[hspace_full.index('2-0')] - eval_tot[hspace_full.index('5-0')])
H0 = qt.Qobj(np.diag(eval_tot))
logi_idx = [hspace_full.index(state) for state in logi_state]
H_qbt_drive = [H0, [n_theta1_dress, ut.drive_gauss_A] ]

num_cpus, n_job = 100, 1
max_steps = 1e-4

tg, drive_amp, detune = params[0] # Independent arguments that can be optimized over
pulse_args = {'drive_amp_A': drive_amp,
            'drive_freq_A': W_20_50 + 2*np.pi*detune,
            'gate_time': tg}
tlist = np.linspace(0, tg, num=3*int(tg))  # total time
options =qt.Options(max_step=max_steps, nsteps=1e4, num_cpus=1 )
print(hspace_full)
print(logi_idx)

['0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1', '5-0', '1-2', '0-8', '2-2']
[0, 3, 4, 13]


In [18]:
c_op_list = []
p = qt.propagator( H=H_qbt_drive,
                        t=tlist,
                        c_op_list=c_op_list,
                        options=options,
                        args=pulse_args,
                        num_cpus=num_cpus,
                        parallel=True,
                        )[-1]  # get the propagator at the final time step
p0_kraus = qt.to_kraus(qt.to_super(p))
p0_kraus = [ut.truncate_2(i, logi_idx) for i in p0_kraus]
p0_kraus_zz = ut.cz_phase_correct(p0_kraus)
p0_super_2 = qt.kraus_to_super(p0_kraus_zz)
f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=cz_gate())
print(f'c_op_list.shape = {np.shape(c_op_list)}', ', num_cpus =', num_cpus)
print('max_steps =', max_steps)
print('fidelity (qutip) =', np.round(np.log10(1-f_noise), 10))

args = [H_qbt_drive, W_20_50, max_steps, num_cpus, c_op_list, logi_idx ]
f_ideal = Parallel(n_jobs=n_job, verbose=0)(delayed(ut.cz_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('fidelity (ZL) =', np.round(f_ideal[0], 10))

c_op_list.shape = (0,) , num_cpus = 100
max_steps = 0.0001
fidelity (qutip) = -0.5419679262
fidelity (ZL) = -0.5419679262


In [19]:
num_cpus = 100
c_op_list = jump_t1_list + jump_tphi_list
p = qt.propagator( H=H_qbt_drive,
                        t=tlist,
                        c_op_list=c_op_list,
                        options=options,
                        args=pulse_args,
                        num_cpus=num_cpus,
                        parallel=True,
                        )[-1]  # get the propagator at the final time step
p0_kraus = qt.to_kraus(qt.to_super(p))
p0_kraus = [ut.truncate_2(i, logi_idx) for i in p0_kraus]
p0_kraus_zz = ut.cz_phase_correct(p0_kraus)
p0_super_2 = qt.kraus_to_super(p0_kraus_zz)
f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=cz_gate())
print(f'c_op_list.shape = {np.shape(c_op_list)}', ', num_cpus =', num_cpus)
print('max_steps =', max_steps)
print('fidelity (qutip) =', np.round(np.log10(1-f_noise), 10))

args = [H_qbt_drive, W_20_50, max_steps, num_cpus, c_op_list, logi_idx ]
f_ideal = Parallel(n_jobs=n_job, verbose=0)(delayed(ut.cz_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('fidelity (ZL) =', np.round(f_ideal[0], 10))

c_op_list.shape = (614, 14, 14) , num_cpus = 100
max_steps = 0.0001
fidelity (qutip) = -0.54195942
fidelity (ZL) = -0.5419603152


In [ ]:
num_cpus = 1
c_op_list = jump_t1_list + jump_tphi_list
p = qt.propagator( H=H_qbt_drive,
                        t=tlist,
                        c_op_list=c_op_list,
                        options=options,
                        args=pulse_args,
                        num_cpus=num_cpus,
                        parallel=True,
                        )[-1]  # get the propagator at the final time step
p0_kraus = qt.to_kraus(qt.to_super(p))
p0_kraus = [ut.truncate_2(i, logi_idx) for i in p0_kraus]
p0_kraus_zz = ut.cz_phase_correct(p0_kraus)
p0_super_2 = qt.kraus_to_super(p0_kraus_zz)
f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=cz_gate())
print(f'c_op_list.shape = {np.shape(c_op_list)}', ', num_cpus =', num_cpus)
print('max_steps =', max_steps)
print('fidelity (qutip) =', np.round(np.log10(1-f_noise), 10))

args = [H_qbt_drive, W_20_50, max_steps, num_cpus, c_op_list, logi_idx ]
f_ideal = Parallel(n_jobs=n_job, verbose=0)(delayed(ut.cz_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('fidelity (ZL) =', np.round(f_ideal[0], 10))